# AmbiCode-Eval Benchmark Demo

This notebook demonstrates how to load, explore, and use the AmbiCode-Eval benchmark.

**AmbiCode-Eval** measures how LLMs handle linguistically ambiguous coding prompts by comparing performance on clean prompts (baseline) vs. ambiguous prompts (experimental condition).

## 1. Loading the Benchmark

In [1]:
import json
import sys
sys.path.insert(0, '..')

from src.data.model import BenchmarkItem

with open('../data/benchmark/benchmark.jsonl') as f:
    items = [BenchmarkItem.from_dict(json.loads(line)) for line in f if line.strip()]

print(f'Loaded {len(items)} benchmark items')

Loaded 62 benchmark items


## 2. Benchmark Distribution

In [2]:
from collections import Counter

# By ambiguity type
type_counts = Counter(it.ambiguity_type for it in items)
print('By ambiguity type:')
for t, c in type_counts.most_common():
    print(f'  {t:25s}: {c}')

# By risk level
print(f'\nBy risk level:')
risk_counts = Counter(it.risk_level for it in items)
for r, c in risk_counts.most_common():
    print(f'  {r:10s}: {c}')

# By source
print(f'\nBy source:')
source_counts = Counter(it.source for it in items)
for s, c in source_counts.most_common():
    print(f'  {s:10s}: {c}')

By ambiguity type:
  collective_distributive  : 19
  syntactic                : 14
  coreferential            : 11
  elliptical               : 10
  scopal                   : 8

By risk level:
  low       : 46
  high      : 16

By source:
  ds1000    : 36
  mbpp      : 26


## 3. Exploring a Benchmark Item

Each item has 3 layers: anchor (baseline), perturbation (experimental), and quality gates.

In [3]:
# Pick an MBPP example (simple format)
item = next(it for it in items if it.source == 'mbpp')

print(f'Task ID:        {item.task_id}')
print(f'Anchor:         {item.anchor_task_id}')
print(f'Ambiguity type: {item.ambiguity_type}')
print(f'Risk level:     {item.risk_level}')

Task ID:        AMBI/001
Anchor:         MBPP/106
Ambiguity type: collective_distributive
Risk level:     low


In [4]:
# Layer 1: Clean prompt (baseline condition)
print('=== CLEAN PROMPT (sent to model for baseline) ===')
print(item.prompt)

=== CLEAN PROMPT (sent to model for baseline) ===
Write a function to append the given list to the given tuples.


In [5]:
# Layer 2: Perturbed prompt (experimental condition)
print('=== PERTURBED PROMPT (sent to model for experimental) ===')
print(item.perturbed_prompt)
print(f'\n--- Interpretation A: {item.interpretation_a}')
print(f'--- Interpretation B: {item.interpretation_b}')

=== PERTURBED PROMPT (sent to model for experimental) ===
def add_lists(test_list, test_tup):
    """
    Write a function to append the given list to the provided tuples.
    """

--- Interpretation A: Append the list's elements to the collection of tuples as a whole, extending the outer sequence.
--- Interpretation B: Distribute the operation by appending the list's elements to each individual tuple contained within the collection.


In [6]:
# Reference solutions for both interpretations
print('=== REF SOLUTION A (interpretation A) ===')
print(item.ref_solution_a)
print('\n=== REF SOLUTION B (interpretation B) ===')
print(item.ref_solution_b)

=== REF SOLUTION A (interpretation A) ===
def add_lists(test_list, test_tup):
  res = tuple(list(test_tup) + test_list)
  return (res) 

=== REF SOLUTION B (interpretation B) ===
def add_lists(test_list, test_tup):
    return tuple(t + tuple(test_list) for t in test_tup)


In [7]:
# Discriminative test suites
print('=== TEST A (passes ref_a, fails ref_b) ===')
print(item.test_a)
print('\n=== TEST B (passes ref_b, fails ref_a) ===')
print(item.test_b)

=== TEST A (passes ref_a, fails ref_b) ===
assert add_lists([5, 6, 7], (9, 10)) == (9, 10, 5, 6, 7)
assert add_lists([6, 7, 8], (10, 11)) == (10, 11, 6, 7, 8)
assert add_lists([7, 8, 9], (11, 12)) == (11, 12, 7, 8, 9)

=== TEST B (passes ref_b, fails ref_a) ===
assert add_lists([5, 6], ((1, 2), (3, 4))) == ((1, 2, 5, 6), (3, 4, 5, 6))
assert add_lists([1], ((0, 0), (1, 1))) == ((0, 0, 1), (1, 1, 1))
assert add_lists([9, 10], ((1,), (2,), (3,))) == ((1, 9, 10), (2, 9, 10), (3, 9, 10))


## 4. Verifying Exclusivity

Every benchmark item guarantees mutual exclusivity: each reference solution passes only its own test suite.

In [8]:
# Verify with a simple MBPP item (no Docker needed)
mbpp_item = next(it for it in items if it.source == 'mbpp')

def run_locally(code, test):
    """Run code + test via exec. Returns True if no exception."""
    try:
        exec(code + '\n' + test, {})
        return True
    except Exception as e:
        return False

ra_ta = run_locally(mbpp_item.ref_solution_a, mbpp_item.test_a)
ra_tb = run_locally(mbpp_item.ref_solution_a, mbpp_item.test_b)
rb_ta = run_locally(mbpp_item.ref_solution_b, mbpp_item.test_a)
rb_tb = run_locally(mbpp_item.ref_solution_b, mbpp_item.test_b)

print(f'{mbpp_item.task_id} ({mbpp_item.ambiguity_type}):')
print(f'  ref_a x test_a: {"PASS" if ra_ta else "FAIL"} (expected PASS)')
print(f'  ref_a x test_b: {"PASS" if ra_tb else "FAIL"} (expected FAIL)')
print(f'  ref_b x test_a: {"PASS" if rb_ta else "FAIL"} (expected FAIL)')
print(f'  ref_b x test_b: {"PASS" if rb_tb else "FAIL"} (expected PASS)')
print(f'  Exclusivity: {"VERIFIED" if (ra_ta and not ra_tb and not rb_ta and rb_tb) else "FAILED"}')

AMBI/001 (collective_distributive):
  ref_a x test_a: PASS (expected PASS)
  ref_a x test_b: FAIL (expected FAIL)
  ref_b x test_a: FAIL (expected FAIL)
  ref_b x test_b: PASS (expected PASS)
  Exclusivity: VERIFIED


## 5. Exploring a DS-1000 Item

DS-1000 items use a normalized format where the solution is wrapped as a `__SOLUTION__` string variable and the test code is the original DS-1000 harness.

In [9]:
ds_item = next(it for it in items if it.source == 'ds1000')

print(f'Task ID:        {ds_item.task_id}')
print(f'Anchor:         {ds_item.anchor_task_id}')
print(f'Library:        {ds_item.library}')
print(f'Ambiguity type: {ds_item.ambiguity_type}')
print(f'Risk level:     {ds_item.risk_level}')
print(f'\n=== PROMPT (first 300 chars) ===')
print(ds_item.prompt[:300])
print(f'\n=== PERTURBED PROMPT (first 300 chars) ===')
print(ds_item.perturbed_prompt[:300])
print(f'\n--- Interpretation A: {ds_item.interpretation_a}')
print(f'--- Interpretation B: {ds_item.interpretation_b}')

Task ID:        AMBI/015
Anchor:         DS1000/Pandas/227
Library:        Pandas
Ambiguity type: coreferential
Risk level:     low

=== PROMPT (first 300 chars) ===
Problem:
i need to create a dataframe containing tuples from a series of dataframes arrays. What I need is the following:
I have dataframes a and b:
a = pd.DataFrame(np.array([[1, 2],[3, 4]]), columns=['one', 'two'])
b = pd.DataFrame(np.array([[5, 6],[7, 8]]), columns=['one', 'two'])
c = pd.DataFram

=== PERTURBED PROMPT (first 300 chars) ===
Problem:
i need to create a dataframe containing tuples from a series of dataframes arrays. What I need is the following:
I have a base dataframe a representing original values, and dataframes b and c representing updates:
a = pd.DataFrame(np.array([[1, 2],[3, 4]]), columns=['one', 'two'])
b = pd.Da

--- Interpretation A: 'them' refers to all three dataframes (a, b, and c), producing a dataframe where each element is a tuple of (a_val, b_val, c_val).
--- Interpretation B: 'them' refer

In [10]:
# DS-1000 canonical_solution is a __SOLUTION__ wrapper
print('=== CANONICAL SOLUTION (normalized format) ===')
print(ds_item.canonical_solution[:200])
print('...')
print(f'\n=== REF SOLUTION B (self-contained) ===')
print(ds_item.ref_solution_b[:300])
print('...')

=== CANONICAL SOLUTION (normalized format) ===
__SOLUTION__ = r"""def g(a,b,c):
    return pd.DataFrame(np.rec.fromarrays((a.values, b.values, c.values)).tolist(),columns=a.columns,index=a.index)

result = g(a.copy(),b.copy(), c.copy())
"""

...

=== REF SOLUTION B (self-contained) ===
import pandas as pd
import numpy as np

a = pd.DataFrame(np.array([[1, 2],[3, 4]]), columns=['one', 'two'])
b = pd.DataFrame(np.array([[5, 6],[7, 8]]), columns=['one', 'two'])
c = pd.DataFrame(np.array([[9, 10],[11, 12]]), columns=['one', 'two'])

def g(a, b, c):
    return pd.DataFrame(np.rec.froma
...


## 6. Using the Benchmark for Evaluation

The standard evaluation workflow for Phase 2+:

In [11]:
# Pseudocode for the full evaluation pipeline
"""
from src.util.llm import LLMClient, ModelConfig
from src.util.sandbox import Sandbox

client = LLMClient()
sandbox = Sandbox()  # or Sandbox(image='ambicode-ds1000') for DS-1000

TARGET_MODELS = ['gpt-5.4', 'claude-sonnet', 'gemini-3.1-pro', 'deepseek-v3.2', 'qwen-3.5']

for model in TARGET_MODELS:
    config = ModelConfig(model=model, temperature=0.8, n=10)
    
    for item in items:
        # 1. Baseline: clean prompt
        clean_responses = client.call(config, prompt=item.prompt)
        
        # 2. Experimental: perturbed prompt
        perturbed_responses = client.call(config, prompt=item.perturbed_prompt)
        
        # 3. Execute clean responses against original tests
        for code in clean_responses.choices:
            result = sandbox.run(code, item.test_code)
            # result.passed -> contributes to pass@k(baseline)
        
        # 4. Execute perturbed responses against both test suites
        for code in perturbed_responses.choices:
            result_a, result_b = sandbox.run_dual_blind(
                code=code,
                test_a=item.test_a,
                test_b=item.test_b,
            )
            # result_a.passed -> pass@k(A)
            # result_b.passed -> pass@k(B)
        
        # 5. Classify behavior: SA / EA / AC
        # (use LLM-as-Judge or regex-based classifier)
        
        # 6. Compute metrics
        # Ambiguity Tax = pass@k(baseline) - pass@k(perturbed)
"""
print('See docs/benchmark_guide.md for full evaluation protocol')

See docs/benchmark_guide.md for full evaluation protocol


## 7. Filtering Items

In [12]:
# Filter by ambiguity type
coref_items = [it for it in items if it.ambiguity_type == 'coreferential']
print(f'Coreferential items: {len(coref_items)}')

# Filter by risk level
high_risk = [it for it in items if it.risk_level == 'high']
print(f'High-risk items: {len(high_risk)}')

# Filter by source
ds1000_items = [it for it in items if it.source == 'ds1000']
print(f'DS-1000 items: {len(ds1000_items)}')

# Filter by library
pandas_items = [it for it in items if it.library == 'Pandas']
print(f'Pandas items: {len(pandas_items)}')

# Cross-filter: high-risk scopal
scopal_high = [it for it in items if it.ambiguity_type == 'scopal' and it.risk_level == 'high']
print(f'Scopal + high-risk items: {len(scopal_high)}')

Coreferential items: 11
High-risk items: 16
DS-1000 items: 36
Pandas items: 22
Scopal + high-risk items: 3


## 8. Summary Statistics

In [13]:
# Cross-tabulation: ambiguity type x risk level
types = ['coreferential', 'syntactic', 'scopal', 'collective_distributive', 'elliptical']

print(f'{"Type":<28s} {"Low":>5s} {"High":>5s} {"Total":>6s}')
print('-' * 48)
for t in types:
    low = sum(1 for it in items if it.ambiguity_type == t and it.risk_level == 'low')
    high = sum(1 for it in items if it.ambiguity_type == t and it.risk_level == 'high')
    print(f'{t:<28s} {low:>5d} {high:>5d} {low+high:>6d}')
print('-' * 48)
total_low = sum(1 for it in items if it.risk_level == 'low')
total_high = sum(1 for it in items if it.risk_level == 'high')
print(f'{"TOTAL":<28s} {total_low:>5d} {total_high:>5d} {len(items):>6d}')

# Field length statistics
print(f'\nAverage field lengths:')
print(f'  prompt:           {sum(len(it.prompt) for it in items)/len(items):.0f} chars')
print(f'  perturbed_prompt: {sum(len(it.perturbed_prompt) for it in items)/len(items):.0f} chars')
print(f'  ref_solution_b:   {sum(len(it.ref_solution_b) for it in items)/len(items):.0f} chars')
print(f'  test_b:           {sum(len(it.test_b) for it in items)/len(items):.0f} chars')

Type                           Low  High  Total
------------------------------------------------
coreferential                    8     3     11
syntactic                       10     4     14
scopal                           5     3      8
collective_distributive         16     3     19
elliptical                       7     3     10
------------------------------------------------
TOTAL                           46    16     62

Average field lengths:
  prompt:           686 chars
  perturbed_prompt: 589 chars
  ref_solution_b:   414 chars
  test_b:           981 chars
